In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

SNAPSHOT_CLEAN_DIR = PROJECT_ROOT / "data" / "snapshot_data" / "training_clean"
EVENT_CLEAN_DIR = PROJECT_ROOT / "data" / "event_data" / "clean"

matches = pd.read_parquet(SNAPSHOT_CLEAN_DIR / "training_matches_clean.parquet")
deaths = pd.read_parquet(EVENT_CLEAN_DIR / "deaths_clean.parquet")
objectives = pd.read_parquet(EVENT_CLEAN_DIR / "objectives_clean.parquet")
mid_boss = pd.read_parquet(EVENT_CLEAN_DIR / "mid_boss_clean.parquet")

print("matches:", matches.shape)
print("deaths:", deaths.shape)
print("objectives:", objectives.shape)
print("mid_boss:", mid_boss.shape)

matches: (120000, 18)
deaths: (845491, 12)
objectives: (265547, 7)
mid_boss: (9964, 4)


In [2]:
exploded_rows = []

for _, row in matches.iterrows():
    ts_time = row['ts_time']
    ts_net_worth = row['ts_net_worth']
    ts_kills = row['ts_kills']
    ts_deaths = row['ts_deaths']
    ts_player_damage = row['ts_player_damage']

    for i in range(len(ts_time)):
        exploded_rows.append({
            'match_id': row['match_id'],
            'account_id': row['account_id'],
            'team': row['team'],
            'won': row['won'],
            'duration_s': row['duration_s'],
            'game_time_s': ts_time[i],
            'net_worth': ts_net_worth[i],
            'kills': ts_kills[i],
            'deaths': ts_deaths[i],
            'player_damage': ts_player_damage[i],
        })

player_timeseries = pd.DataFrame(exploded_rows)
print(f"Exploded player timeseries: {player_timeseries.shape[0]} rows")
player_timeseries.head()

Exploded player timeseries: 1071972 rows


,match_id,account_id,team,won,duration_s,game_time_s,net_worth,kills,deaths,player_damage
0,25313355,62046419,Team1,True,2460,180,1525,0,0,1085
1,25313355,62046419,Team1,True,2460,360,3553,0,1,1717
2,25313355,62046419,Team1,True,2460,540,5452,0,1,1756
3,25313355,62046419,Team1,True,2460,720,7653,0,2,2080
4,25313355,62046419,Team1,True,2460,900,10992,0,2,2563


In [3]:
team_timeseries = (
    player_timeseries
    .groupby(['match_id', 'team', 'game_time_s', 'won', 'duration_s'])
    .agg(
        team_net_worth=('net_worth', 'sum'),
        team_kills=('kills', 'sum'),
        team_deaths=('deaths', 'sum'),
        team_damage=('player_damage', 'sum'),
    )
    .reset_index()
)

print(f"Team timeseries: {team_timeseries.shape[0]} rows")
team_timeseries.head(10)

Team timeseries: 178662 rows


,match_id,team,game_time_s,won,duration_s,team_net_worth,team_kills,team_deaths,team_damage
0,16000346,Team0,180,False,2068,7592,1,0,4968
1,16000346,Team0,360,False,2068,19669,4,6,11879
2,16000346,Team0,540,False,2068,37981,11,10,20806
3,16000346,Team0,720,False,2068,49211,15,17,33423
4,16000346,Team0,900,False,2068,64792,19,21,47023
5,16000346,Team0,1200,False,2068,97337,23,33,77631
6,16000346,Team0,1500,False,2068,142279,30,37,111210
7,16000346,Team0,1800,False,2068,178113,35,47,141598
8,16000346,Team0,2068,False,2068,197864,38,54,167146
9,16000346,Team1,180,True,2068,9264,0,3,3794


In [4]:
# faster vectorized version? instead of a nested loop

import numpy as np

def add_cumulative_deaths_fast(team_timeseries, deaths):
    deaths_sorted = deaths.sort_values(['match_id', 'team', 'game_time_s']).copy()
    deaths_sorted['game_time_s'] = deaths_sorted['game_time_s'].astype('int64')
    deaths_sorted['death_count'] = 1
    deaths_sorted['cumulative_deaths'] = (
        deaths_sorted.groupby(['match_id', 'team'])['death_count'].cumsum()
    )

    team_timeseries = team_timeseries.copy()
    team_timeseries['game_time_s'] = team_timeseries['game_time_s'].astype('int64')

    results = []

    for team_label in ['Team0', 'Team1']:
        enemy_label = 'Team1' if team_label == 'Team0' else 'Team0'

        own_deaths = deaths_sorted[deaths_sorted['team'] == team_label][
            ['match_id', 'game_time_s', 'cumulative_deaths']
        ].rename(columns={'cumulative_deaths': 'own_deaths_cumulative'})

        enemy_deaths = deaths_sorted[deaths_sorted['team'] == enemy_label][
            ['match_id', 'game_time_s', 'cumulative_deaths']
        ].rename(columns={'cumulative_deaths': 'enemy_deaths_cumulative'})

        ts_subset = team_timeseries[team_timeseries['team'] == team_label].sort_values('game_time_s')

        merged = pd.merge_asof(
            ts_subset.sort_values('game_time_s'),
            own_deaths.sort_values('game_time_s'),
            by='match_id', on='game_time_s', direction='backward'
        )
        merged = pd.merge_asof(
            merged.sort_values('game_time_s'),
            enemy_deaths.sort_values('game_time_s'),
            by='match_id', on='game_time_s', direction='backward'
        )

        results.append(merged)

    out = pd.concat(results, ignore_index=True)
    out['own_deaths_cumulative'] = out['own_deaths_cumulative'].fillna(0)
    out['enemy_deaths_cumulative'] = out['enemy_deaths_cumulative'].fillna(0)
    return out

team_timeseries_with_deaths = add_cumulative_deaths_fast(team_timeseries, deaths)
print(team_timeseries_with_deaths.shape)
team_timeseries_with_deaths.head(10)

(178662, 11)


,match_id,team,game_time_s,won,duration_s,team_net_worth,team_kills,team_deaths,team_damage,own_deaths_cumulative,enemy_deaths_cumulative
0,27533595,Team0,180,True,1599,9654,2,1,4250,1.0,2.0
1,16024102,Team0,180,True,1613,8939,1,1,4495,1.0,1.0
2,16023536,Team0,180,False,1879,5475,1,1,3019,1.0,1.0
3,16086878,Team0,180,True,1935,7264,2,1,3428,1.0,2.0
4,16083088,Team0,180,True,999,10248,9,1,6075,1.0,8.0
5,26602595,Team0,180,True,2001,8768,2,1,4609,1.0,3.0
6,22862928,Team0,180,True,2296,9551,1,4,3611,4.0,1.0
7,22862900,Team0,180,True,2837,8436,0,2,2860,2.0,0.0
8,25909814,Team0,180,False,2158,9745,4,3,6035,3.0,4.0
9,25909812,Team0,180,False,2231,10589,2,1,5474,1.0,2.0


In [5]:
def add_cumulative_objectives(team_timeseries, objectives):
    obj_sorted = objectives.sort_values(['match_id', 'team', 'destroyed_time_s']).copy()
    obj_sorted['destroyed_time_s'] = obj_sorted['destroyed_time_s'].astype('int64')
    obj_sorted['obj_count'] = 1
    obj_sorted['cumulative_objectives'] = (
        obj_sorted.groupby(['match_id', 'team'])['obj_count'].cumsum()
    )

    team_timeseries = team_timeseries.copy()

    results = []

    for team_label in ['Team0', 'Team1']:
        enemy_label = 'Team1' if team_label == 'Team0' else 'Team0'

        own_obj = obj_sorted[obj_sorted['team'] == team_label][
            ['match_id', 'destroyed_time_s', 'cumulative_objectives']
        ].rename(columns={'destroyed_time_s': 'game_time_s', 'cumulative_objectives': 'own_objectives_cumulative'})

        enemy_obj = obj_sorted[obj_sorted['team'] == enemy_label][
            ['match_id', 'destroyed_time_s', 'cumulative_objectives']
        ].rename(columns={'destroyed_time_s': 'game_time_s', 'cumulative_objectives': 'enemy_objectives_cumulative'})

        ts_subset = team_timeseries[team_timeseries['team'] == team_label].sort_values('game_time_s')

        merged = pd.merge_asof(
            ts_subset.sort_values('game_time_s'),
            own_obj.sort_values('game_time_s'),
            by='match_id', on='game_time_s', direction='backward'
        )
        merged = pd.merge_asof(
            merged.sort_values('game_time_s'),
            enemy_obj.sort_values('game_time_s'),
            by='match_id', on='game_time_s', direction='backward'
        )

        results.append(merged)

    out = pd.concat(results, ignore_index=True)
    out['own_objectives_cumulative'] = out['own_objectives_cumulative'].fillna(0)
    out['enemy_objectives_cumulative'] = out['enemy_objectives_cumulative'].fillna(0)
    return out


def add_mid_boss_flag(team_timeseries, mid_boss):
    mb_sorted = mid_boss.sort_values(['match_id', 'destroyed_time_s']).copy()
    mb_sorted['destroyed_time_s'] = mb_sorted['destroyed_time_s'].astype('int64')

    team_timeseries = team_timeseries.copy()

    results = []

    for team_label in ['Team0', 'Team1']:
        own_mb = mb_sorted[mb_sorted['team_claimed'] == team_label][
            ['match_id', 'destroyed_time_s']
        ].copy()
        own_mb['own_mid_boss_claims'] = own_mb.groupby('match_id').cumcount() + 1
        own_mb = own_mb.rename(columns={'destroyed_time_s': 'game_time_s'})

        enemy_label = 'Team1' if team_label == 'Team0' else 'Team0'
        enemy_mb = mb_sorted[mb_sorted['team_claimed'] == enemy_label][
            ['match_id', 'destroyed_time_s']
        ].copy()
        enemy_mb['enemy_mid_boss_claims'] = enemy_mb.groupby('match_id').cumcount() + 1
        enemy_mb = enemy_mb.rename(columns={'destroyed_time_s': 'game_time_s'})

        ts_subset = team_timeseries[team_timeseries['team'] == team_label].sort_values('game_time_s')

        merged = pd.merge_asof(
            ts_subset.sort_values('game_time_s'),
            own_mb[['match_id', 'game_time_s', 'own_mid_boss_claims']].sort_values('game_time_s'),
            by='match_id', on='game_time_s', direction='backward'
        )
        merged = pd.merge_asof(
            merged.sort_values('game_time_s'),
            enemy_mb[['match_id', 'game_time_s', 'enemy_mid_boss_claims']].sort_values('game_time_s'),
            by='match_id', on='game_time_s', direction='backward'
        )

        results.append(merged)

    out = pd.concat(results, ignore_index=True)
    out['own_mid_boss_claims'] = out['own_mid_boss_claims'].fillna(0)
    out['enemy_mid_boss_claims'] = out['enemy_mid_boss_claims'].fillna(0)
    return out

features = add_cumulative_objectives(team_timeseries_with_deaths, objectives)
features = add_mid_boss_flag(features, mid_boss)

print(features.shape)
features.head(10)

(178662, 15)


,match_id,team,game_time_s,won,duration_s,team_net_worth,team_kills,team_deaths,team_damage,own_deaths_cumulative,enemy_deaths_cumulative,own_objectives_cumulative,enemy_objectives_cumulative,own_mid_boss_claims,enemy_mid_boss_claims
0,27533595,Team0,180,True,1599,9654,2,1,4250,1.0,2.0,1.0,0.0,0.0,0.0
1,16024102,Team0,180,True,1613,8939,1,1,4495,1.0,1.0,3.0,0.0,0.0,0.0
2,16023536,Team0,180,False,1879,5475,1,1,3019,1.0,1.0,0.0,2.0,0.0,0.0
3,16086878,Team0,180,True,1935,7264,2,1,3428,1.0,2.0,1.0,0.0,0.0,0.0
4,16083088,Team0,180,True,999,10248,9,1,6075,1.0,8.0,1.0,0.0,0.0,0.0
5,26602595,Team0,180,True,2001,8768,2,1,4609,1.0,3.0,1.0,0.0,0.0,0.0
6,22862928,Team0,180,True,2296,9551,1,4,3611,4.0,1.0,3.0,1.0,0.0,0.0
7,22862900,Team0,180,True,2837,8436,0,2,2860,2.0,0.0,2.0,1.0,0.0,0.0
8,25909814,Team0,180,False,2158,9745,4,3,6035,3.0,4.0,0.0,3.0,0.0,0.0
9,25909812,Team0,180,False,2231,10589,2,1,5474,1.0,2.0,0.0,2.0,0.0,0.0


In [6]:
FEATURE_DIR = PROJECT_ROOT / "data" / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

features.to_parquet(FEATURE_DIR / "team_timeseries_features.parquet")
print(f"Saved {features.shape[0]} rows to {FEATURE_DIR / 'team_timeseries_features.parquet'}")

Saved 178662 rows to C:\.STUFF\coding\dl-match-analyser\data\features\team_timeseries_features.parquet


In [9]:
stat_cols_team0 = [f'{c}_team0' for c in ['team_net_worth', 'team_kills', 'team_deaths', 'team_damage']]
stat_cols_team1 = [f'{c}_team1' for c in ['team_net_worth', 'team_kills', 'team_deaths', 'team_damage']]

for col in stat_cols_team0 + stat_cols_team1:
    merged[col] = merged[col].astype('int64')

for col in ['own_objectives_cumulative', 'enemy_objectives_cumulative',
            'own_deaths_cumulative', 'enemy_deaths_cumulative',
            'own_mid_boss_claims', 'enemy_mid_boss_claims']:
    merged[col] = merged[col].astype('int64')

merged['net_worth_diff'] = merged['team_net_worth_team0'] - merged['team_net_worth_team1']
merged['kills_diff'] = merged['team_kills_team0'] - merged['team_kills_team1']
merged['deaths_diff'] = merged['team_deaths_team0'] - merged['team_deaths_team1']
merged['damage_diff'] = merged['team_damage_team0'] - merged['team_damage_team1']
merged['objectives_diff'] = merged['own_objectives_cumulative'] - merged['enemy_objectives_cumulative']
merged['deaths_cumulative_diff'] = merged['own_deaths_cumulative'] - merged['enemy_deaths_cumulative']
merged['mid_boss_diff'] = merged['own_mid_boss_claims'] - merged['enemy_mid_boss_claims']

print(merged.shape)
merged[['match_id', 'game_time_s', 'won', 'net_worth_diff', 'kills_diff',
        'deaths_diff', 'objectives_diff', 'mid_boss_diff']].head(10)

(89331, 26)


,match_id,game_time_s,won,net_worth_diff,kills_diff,deaths_diff,objectives_diff,mid_boss_diff
0,27533595,180,True,-287,1,-1,1,0
1,16024102,180,True,991,0,0,3,0
2,16023536,180,False,-153,1,0,-2,0
3,16086878,180,True,-261,1,-1,1,0
4,16083088,180,True,6358,9,-8,1,0
5,26602595,180,True,724,1,-2,1,0
6,22862928,180,True,-936,-3,3,2,0
7,22862900,180,True,147,-1,2,1,0
8,25909814,180,False,1723,2,-1,-3,0
9,25909812,180,False,968,1,-1,-2,0


In [11]:
merged.to_parquet(FEATURE_DIR / "team_diff_features.parquet")
print(f"Saved {merged.shape[0]} rows to {FEATURE_DIR / 'team_diff_features.parquet'}")

Saved 89331 rows to C:\.STUFF\coding\dl-match-analyser\data\features\team_diff_features.parquet


In [13]:
# mirrored data from the other teams perspective
mirrored = merged.copy()

diff_cols = ['net_worth_diff', 'kills_diff', 'deaths_diff', 'damage_diff',
             'objectives_diff', 'deaths_cumulative_diff', 'mid_boss_diff']

for col in diff_cols:
    mirrored[col] = -mirrored[col]

mirrored['won'] = ~mirrored['won']

merged['perspective'] = 'Team0'
mirrored['perspective'] = 'Team1'

full_features = pd.concat([merged, mirrored], ignore_index=True)

print(f"Combined table: {full_features.shape[0]} rows (was {merged.shape[0]} before mirroring)")
full_features[['match_id', 'game_time_s', 'perspective', 'won', 'net_worth_diff',
               'kills_diff', 'objectives_diff', 'mid_boss_diff']].head(10)

Combined table: 178662 rows (was 89331 before mirroring)


,match_id,game_time_s,perspective,won,net_worth_diff,kills_diff,objectives_diff,mid_boss_diff
0,27533595,180,Team0,True,-287,1,1,0
1,16024102,180,Team0,True,991,0,3,0
2,16023536,180,Team0,False,-153,1,-2,0
3,16086878,180,Team0,True,-261,1,1,0
4,16083088,180,Team0,True,6358,9,1,0
5,26602595,180,Team0,True,724,1,1,0
6,22862928,180,Team0,True,-936,-3,2,0
7,22862900,180,Team0,True,147,-1,1,0
8,25909814,180,Team0,False,1723,2,-3,0
9,25909812,180,Team0,False,968,1,-2,0


In [16]:
full_features.to_parquet(FEATURE_DIR / "full_features_mirrored.parquet")
print(f"Saved {full_features.shape[0]} rows to {FEATURE_DIR / 'full_features_mirrored.parquet'}")

Saved 178662 rows to C:\.STUFF\coding\dl-match-analyser\data\features\full_features_mirrored.parquet
